In [12]:
!pip install cma

     |████████████████████████████████| 249 kB 4.1 MB/s 


In [13]:
!git clone --recursive https://github.com/hardmaru/estool.git

Cloning into 'estool'...
remote: Enumerating objects: 418, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 418 (delta 71), reused 63 (delta 63), pack-reused 317
Receiving objects: 100% (418/418), 7.29 MiB | 8.35 MiB/s, done.
Resolving deltas: 100% (165/165), done.


In [14]:
import numpy as np
import gym
import cma
from estool.es import SimpleGA, CMAES, PEPG, OpenES

In [7]:
class Agent:
  def __init__(self, q_sol, env_size = (10, 10)):
    self.q_table = np.reshape(q_sol, env_size)
    
  def get_action(self, obs):
    print(self.q_table[obs])
    return np.argmax(self.q_table[obs])

In [8]:
def rollout(agent, env):

  obs = env.reset()

  done = False

  total_reward = 0

  while not done:

    a = agent.get_action(obs)

    next_obs, reward, done, info = env.step(a)

    total_reward += reward

    obs =  next_obs

    # print(obs, a)

  return total_reward

In [15]:
env = gym.make('Taxi-v3')

env_size = (env.observation_space.n, env.action_space.n)

NPARAMS = env.observation_space.n * env.action_space.n
NPOPULATION = 100
MAX_ITERATION = 10000

# defines genetic algorithm solver
ga = SimpleGA(NPARAMS,                # number of model parameters
               sigma_init=5,        # initial standard deviation
               popsize=NPOPULATION,   # population size
               elite_ratio=0.1,       # percentage of the elites
               forget_best=False,     # forget the historical best elites
               weight_decay=0.005,     # weight decay coefficient
              )

# defines CMA-ES algorithm solver
cmaes = CMAES(NPARAMS,
              popsize=NPOPULATION,
              weight_decay = 1,
              sigma_init = 1
          )


(50_w,100)-aCMA-ES (mu_w=27.0,w_1=8%) in dimension 3000 (seed=769471, Tue May  3 15:52:43 2022)


In [16]:
def test_solver(solver, iter = 100):

  history = []

  for j in range(iter):

    # ask the ES to give set of params
    solutions = solver.ask()

    # create array to hold the results
    fitlist = np.zeros(solver.popsize)

    # evaluate for each given solution
    for i in range(solver.popsize):

      # init the agent with a solution
      agent = Agent(solutions[i], env_size)

      # rollout env with this agent
      fitlist[i] = rollout(agent, env)

    # give scores results back to ES
    solver.tell(fitlist)

    # get best param & fitness from ES
    results = solver.result()

    history.append(results[1])

    if (j+1) % 10 == 0:
        print("fitness at iteration", (j+1), results[1])
        print(fitlist)
        print(results[0])

  return results[0], results[1], history


In [ ]:
# use our favourite ES
bestsol, bestfit, history = test_solver(ga, MAX_ITERATION)
# print(bestsol)

**Referencias**

*   https://gym.openai.com/docs/ 
*   https://lilianweng.github.io/lil-log/2018/02/19/a-long-peek-into-reinforcement-learning.html
*   https://blog.otoro.net/2017/10/29/visual-evolution-strategies/
*   https://blog.otoro.net/2017/11/12/evolving-stable-strategies/
*   https://github.com/hardmaru/estool/
*   https://github.com/CyberAgent/cmaes
*   https://towardsdatascience.com/reinforcement-learning-without-gradients-evolving-agents-using-genetic-algorithms-8685817d84f 
*   https://sites.wustl.edu/jeffheaton/t81-558/ 
*   https://towardsdatascience.com/reinforcement-learning-without-gradients-evolving-agents-using-genetic-algorithms-8685817d84f
*   https://github.com/mymultiverse/GeneticAlgo_OpenAIGymCartPole
*   https://github.com/rlcode/reinforcement-learning 